# Smart Greenhouse ML Preprocessing and Windowing

## 00 - Experiment Overview

Notebook nay bien 24 trajectory synthetic canonical thanh cac sample supervised de dung cho
milestone train model tiep theo. Pipeline dung `full_dataset_index.csv` lam membership duy nhat,
giu scenario va thoi gian tach biet, fit scaler chi tren TRAIN, va tao window theo kieu lazy.

**Input:** 24 file ML canonical, manifest parameter V2 va dataset index da audit.

**Output:** cac `Dataset`/`DataLoader` san sang cho PyTorch va artifact preprocessing co provenance.

> Pham vi dung tai DataLoader. Notebook nay khong train GRU, LSTM hay Transformer.

## 01 - Environment & Configuration

Tat ca tham so co the thay doi duoc dat tai mot noi. `SMOKE_TEST=False` la default da luu;
Codex co the bat smoke mode bang bien moi truong ma khong sua notebook. Sai config o buoc nay co
the lam split, shape hoac duong dan khong con tai lap.

In [ ]:
import os
from pathlib import Path

SEED = 20260816
LOOKBACK_STEPS = 24
FORECAST_HORIZON = 1
BATCH_SIZE = 256
NUM_WORKERS = 0
HELD_OUT_SCENARIO_COUNT = 4

SMOKE_TEST = False
SMOKE_TEST = os.getenv("GREENHOUSE_SMOKE_TEST", str(SMOKE_TEST)).lower() in {
    "1", "true", "yes"
}

default_data_root = Path("/content/smart_greenhouse_dataset") if Path("/content").exists() else Path.cwd()
DATA_ROOT = Path(os.getenv("GREENHOUSE_DATA_ROOT", str(default_data_root))).expanduser()
INDEX_FILE = DATA_ROOT / "full_dataset_index.csv"
PARAMETER_MANIFEST_FILE = DATA_ROOT / "final_approved_parameter_sets_v2.csv"
ARTIFACT_DIR = Path(
    os.getenv(
        "GREENHOUSE_ARTIFACT_DIR",
        str(DATA_ROOT / "artifacts" / ("preprocessing_smoke" if SMOKE_TEST else "preprocessing")),
    )
)

EXPECTED_SCENARIOS = 24
EXPECTED_ROWS_PER_SCENARIO = 70_128
EXPECTED_START = "2018-01-01 00:00:00"
EXPECTED_END = "2025-12-31 23:00:00"

print(f"DATA_ROOT={DATA_ROOT.resolve()}")
print(f"SMOKE_TEST={SMOKE_TEST}, lookback={LOOKBACK_STEPS}, horizon={FORECAST_HORIZON}")

## 02 - Imports

Notebook chi dung cac thu vien pho bien san co tren Google Colab. GPU khong can cho preprocessing;
tensor van duoc tao tren CPU va viec chuyen device se thuoc notebook training sau.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
import json
import platform
import random
import warnings

import joblib
import nbformat
import numpy as np
import pandas as pd
import sklearn
from sklearn.preprocessing import StandardScaler
import torch
from torch.utils.data import DataLoader, Dataset

print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 03 - Reproducibility Setup

Seed chung bao dam viec shuffle window va cac lua chon deterministic co the lap lai. Notebook chua
train GPU nen khong ep cac che do CUDA deterministic co chi phi cao.

In [ ]:
def set_reproducibility(seed: int) -> torch.Generator:
    """Seed Python, NumPy and PyTorch, then return a seeded DataLoader generator."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    generator = torch.Generator()
    generator.manual_seed(seed)
    return generator


data_loader_generator = set_reproducibility(SEED)

## 04 - Dataset Location / Path Configuration

Index duoc sinh tren Windows co the chua dau `\\`. Resolver chuyen path ve POSIX-style truoc khi
ghep voi `DATA_ROOT`; fallback chi thu cac vi tri ro rang va luon phat warning, khong tim mo ho bang glob.

In [ ]:
def normalize_index_path(raw_path: str) -> Path:
    """Normalize a manifest path without assuming the host path separator."""
    normalized = str(raw_path).strip().replace("\\", "/")
    if not normalized:
        raise ValueError("Canonical index contains an empty path")
    return Path(normalized)


def resolve_scenario_path(raw_path: str, data_root: Path) -> Path:
    """Resolve one indexed ML file, with explicit and auditable fallbacks."""
    normalized = normalize_index_path(raw_path)
    candidates = [
        normalized if normalized.is_absolute() else data_root / normalized,
        data_root / "outputs" / "full_generation" / "ml" / normalized.name,
        data_root / "ml" / normalized.name,
        data_root / normalized.name,
    ]
    unique_candidates = list(dict.fromkeys(path.resolve() for path in candidates))
    existing = [path for path in unique_candidates if path.is_file()]
    if not existing:
        raise FileNotFoundError(
            f"Cannot resolve indexed ML file {raw_path!r}; tried: {unique_candidates}"
        )
    preferred = unique_candidates[0]
    if preferred in existing:
        return preferred
    if len(existing) != 1:
        raise RuntimeError(f"Ambiguous fallback for {raw_path!r}: {existing}")
    warnings.warn(
        f"Indexed path {raw_path!r} was resolved by explicit fallback to {existing[0]}",
        RuntimeWarning,
    )
    return existing[0]

## 05 - Load Canonical Dataset Index

`full_dataset_index.csv` la source of truth cho membership. Notebook khong glob thu muc ML, vi thu muc
do co the con artifact bi reject tu V1. Parameter manifest chi bo sung toa do vat ly cho scenario split.

In [ ]:
INDEX_REQUIRED_COLUMNS = {
    "parameter_set_id", "ml_file", "ml_rows", "ml_hash", "config_hash", "validation_status"
}
PARAMETER_COLUMNS = ["C_d", "eta_s", "C_s_J_K", "irrigation_flow_L_h", "ET_scale"]


def load_canonical_dataset_index(index_file: Path) -> pd.DataFrame:
    """Load and validate canonical membership without inspecting unrelated directory entries."""
    if not index_file.is_file():
        raise FileNotFoundError(f"Canonical index not found: {index_file}")
    index = pd.read_csv(index_file)
    missing = INDEX_REQUIRED_COLUMNS.difference(index.columns)
    if missing:
        raise ValueError(f"Canonical index is missing columns: {sorted(missing)}")
    if len(index) != EXPECTED_SCENARIOS:
        raise ValueError(f"Expected {EXPECTED_SCENARIOS} canonical scenarios, found {len(index)}")
    if index["parameter_set_id"].duplicated().any():
        raise ValueError("Canonical index contains duplicate parameter_set_id values")
    if index["config_hash"].duplicated().any():
        raise ValueError("Canonical index contains duplicate parameter configurations")
    if not (index["validation_status"] == "PASS").all():
        raise ValueError("Canonical index includes a scenario without PASS validation")
    if not (index["ml_rows"].astype(int) == EXPECTED_ROWS_PER_SCENARIO).all():
        raise ValueError("Canonical index contains an unexpected ML row count")
    return index.sort_values("parameter_set_id").reset_index(drop=True)


canonical_index = load_canonical_dataset_index(INDEX_FILE)
parameter_manifest = pd.read_csv(PARAMETER_MANIFEST_FILE)
print(f"Canonical scenarios: {len(canonical_index)} (index membership only)")

## 06 - Resolve Canonical Scenario Files

Moi identity canonical duoc map toi dung mot file ton tai. So sanh tap ID giua index va manifest ngan
viec split mot parameter set khong nam trong corpus cuoi.

In [ ]:
def resolve_canonical_scenario_files(index: pd.DataFrame, data_root: Path) -> dict[str, Path]:
    """Resolve exactly the indexed scenario files; never discover membership by globbing."""
    resolved = {
        row.parameter_set_id: resolve_scenario_path(row.ml_file, data_root)
        for row in index.itertuples(index=False)
    }
    if len(set(resolved.values())) != len(resolved):
        raise ValueError("Multiple canonical identities resolve to the same ML file")
    return resolved


scenario_paths = resolve_canonical_scenario_files(canonical_index, DATA_ROOT)
index_ids = set(canonical_index["parameter_set_id"])
manifest_ids = set(parameter_manifest["parameter_set_id"])
if index_ids != manifest_ids:
    raise ValueError(
        f"Index/manifest identity mismatch: index_only={sorted(index_ids - manifest_ids)}, "
        f"manifest_only={sorted(manifest_ids - index_ids)}"
    )
print(f"Resolved {len(scenario_paths)} canonical files")

## 07 - Dataset Integrity Validation

Validation fail-fast: exact schema, 70,128 dong, hourly continuity, leap days, finite values va actuator
nhi phan. Khong `dropna`, fill, clip, noi suy hay sua du lieu ngam.

In [ ]:
SOURCE_COLUMNS = [
    "timestamp",
    "air_temperature",
    "air_humidity",
    "soil_temperature",
    "soil_moisture",
    "light_lux",
    "pump_state",
    "fan_state",
    "grow_light_state",
]
NUMERIC_COLUMNS = SOURCE_COLUMNS[1:]
ACTUATOR_COLUMNS = ["pump_state", "fan_state", "grow_light_state"]


def validate_scenario_dataframe(frame: pd.DataFrame, scenario_id: str) -> pd.DataFrame:
    """Return a timestamp-parsed frame only after all source invariants pass."""
    if list(frame.columns) != SOURCE_COLUMNS:
        raise ValueError(f"{scenario_id}: schema mismatch: {list(frame.columns)}")
    if len(frame) != EXPECTED_ROWS_PER_SCENARIO:
        raise ValueError(f"{scenario_id}: expected 70,128 rows, found {len(frame)}")

    validated = frame.copy()
    validated["timestamp"] = pd.to_datetime(validated["timestamp"], errors="raise")
    if validated["timestamp"].duplicated().any():
        raise ValueError(f"{scenario_id}: duplicate timestamps")
    if not validated["timestamp"].is_monotonic_increasing:
        raise ValueError(f"{scenario_id}: timestamps are not ascending")
    if validated["timestamp"].iloc[0] != pd.Timestamp(EXPECTED_START):
        raise ValueError(f"{scenario_id}: unexpected first timestamp")
    if validated["timestamp"].iloc[-1] != pd.Timestamp(EXPECTED_END):
        raise ValueError(f"{scenario_id}: unexpected last timestamp")
    intervals = validated["timestamp"].diff().dropna()
    if not (intervals == pd.Timedelta(hours=1)).all():
        raise ValueError(f"{scenario_id}: timestamps are not continuous hourly data")
    for leap_day in ("2020-02-29", "2024-02-29"):
        if (validated["timestamp"].dt.strftime("%Y-%m-%d") == leap_day).sum() != 24:
            raise ValueError(f"{scenario_id}: {leap_day} does not contain 24 rows")

    numeric = validated[NUMERIC_COLUMNS].to_numpy(dtype=np.float64)
    if validated[NUMERIC_COLUMNS].isna().any().any():
        raise ValueError(f"{scenario_id}: NaN detected")
    if not np.isfinite(numeric).all():
        raise ValueError(f"{scenario_id}: infinite numeric value detected")
    for actuator in ACTUATOR_COLUMNS:
        if not set(validated[actuator].unique()).issubset({0, 1}):
            raise ValueError(f"{scenario_id}: {actuator} is not binary")
    return validated


def load_validated_scenarios(
    selected_ids: list[str], paths: dict[str, Path]
) -> dict[str, pd.DataFrame]:
    """Load only selected canonical identities and validate each complete source file."""
    return {
        scenario_id: validate_scenario_dataframe(pd.read_csv(paths[scenario_id]), scenario_id)
        for scenario_id in selected_ids
    }

## 08 - Scenario Split Design

Bon held-out scenario duoc chon deterministic bang farthest-point trong khong gian 5 parameter da
normalize. Chung hoan toan khong xuat hien trong TRAIN, giup do scenario generalization thay vi chi nho
mot weather trajectory cu.

In [ ]:
def select_held_out_scenarios(
    manifest: pd.DataFrame,
    count: int = HELD_OUT_SCENARIO_COUNT,
) -> list[str]:
    """Select deterministic, parameter-space-diverse held-out scenarios."""
    required = {"parameter_set_id", *PARAMETER_COLUMNS}
    missing = required.difference(manifest.columns)
    if missing:
        raise ValueError(f"Parameter manifest is missing: {sorted(missing)}")
    ordered = manifest.sort_values("parameter_set_id").reset_index(drop=True)
    values = ordered[PARAMETER_COLUMNS].to_numpy(dtype=np.float64)
    spans = np.ptp(values, axis=0)
    if (spans == 0).any():
        raise ValueError("A parameter dimension has zero span; coverage selection is undefined")
    normalized = (values - values.min(axis=0)) / spans
    centroid = normalized.mean(axis=0)
    ids = ordered["parameter_set_id"].tolist()

    distance_to_centroid = np.linalg.norm(normalized - centroid, axis=1)
    first = sorted(range(len(ids)), key=lambda i: (-distance_to_centroid[i], ids[i]))[0]
    selected = [first]
    while len(selected) < count:
        remaining = [i for i in range(len(ids)) if i not in selected]
        minimum_distance = {
            i: min(np.linalg.norm(normalized[i] - normalized[j]) for j in selected)
            for i in remaining
        }
        next_index = sorted(remaining, key=lambda i: (-minimum_distance[i], ids[i]))[0]
        selected.append(next_index)
    return sorted(ids[index] for index in selected)


held_out_scenario_ids = select_held_out_scenarios(parameter_manifest)
development_scenario_ids = sorted(index_ids.difference(held_out_scenario_ids))
if len(development_scenario_ids) != 20 or len(held_out_scenario_ids) != 4:
    raise AssertionError("Scenario split must contain 20 development and 4 held-out identities")

active_development_ids = development_scenario_ids[:1] if SMOKE_TEST else development_scenario_ids
active_held_out_ids = held_out_scenario_ids[:1] if SMOKE_TEST else held_out_scenario_ids
active_scenario_ids = active_development_ids + active_held_out_ids
scenario_frames = load_validated_scenarios(active_scenario_ids, scenario_paths)

print(f"Development scenarios ({len(development_scenario_ids)}): {development_scenario_ids}")
print(f"Held-out scenarios ({len(held_out_scenario_ids)}): {held_out_scenario_ids}")
print(f"Loaded scenarios in this run: {active_scenario_ids}")

## 09 - Temporal Split Design

Development trajectories dung 2018-2023 cho TRAIN, 2024 cho validation va 2025 cho temporal test.
Smoke mode chi giu cac doan ngan tu ba period, kem lookback context, de kiem thu nhanh ma khong doi
split contract cua full notebook.

In [ ]:
FULL_SPLIT_RANGES = {
    "train": (pd.Timestamp("2018-01-01 00:00"), pd.Timestamp("2023-12-31 23:00")),
    "validation": (pd.Timestamp("2024-01-01 00:00"), pd.Timestamp("2024-12-31 23:00")),
    "temporal_test": (pd.Timestamp("2025-01-01 00:00"), pd.Timestamp("2025-12-31 23:00")),
}
SMOKE_SPLIT_RANGES = {
    "train": (pd.Timestamp("2018-01-01 00:00"), pd.Timestamp("2018-01-14 23:00")),
    "validation": (pd.Timestamp("2024-01-01 00:00"), pd.Timestamp("2024-01-07 23:00")),
    "temporal_test": (pd.Timestamp("2025-01-01 00:00"), pd.Timestamp("2025-01-07 23:00")),
}
active_split_ranges = SMOKE_SPLIT_RANGES if SMOKE_TEST else FULL_SPLIT_RANGES


def retain_smoke_ranges(frame: pd.DataFrame) -> pd.DataFrame:
    """Keep short scored periods plus sufficient historical context for boundary windows."""
    keep = np.zeros(len(frame), dtype=bool)
    context_hours = LOOKBACK_STEPS + FORECAST_HORIZON - 1
    for start, end in active_split_ranges.values():
        context_start = start - pd.Timedelta(hours=context_hours)
        keep |= frame["timestamp"].between(context_start, end).to_numpy()
    return frame.loc[keep].reset_index(drop=True)


if SMOKE_TEST:
    scenario_frames = {
        scenario_id: retain_smoke_ranges(frame)
        for scenario_id, frame in scenario_frames.items()
    }

train_start, train_end = active_split_ranges["train"]
validation_start, validation_end = active_split_ranges["validation"]
temporal_test_start, temporal_test_end = active_split_ranges["temporal_test"]
print({name: (str(start), str(end)) for name, (start, end) in active_split_ranges.items()})

## 10 - Feature / Target Contract

Moi timestep co 8 feature deployment-available; target chi gom 5 trang thai moi truong o tuong lai.
Timestamp va parameter metadata chi phuc vu index/provenance, khong bao gio vao tensor model.

In [ ]:
FEATURE_COLUMNS = [
    "air_temperature",
    "air_humidity",
    "soil_temperature",
    "soil_moisture",
    "light_lux",
    "pump_state",
    "fan_state",
    "grow_light_state",
]
TARGET_COLUMNS = [
    "air_temperature",
    "air_humidity",
    "soil_temperature",
    "soil_moisture",
    "light_lux",
]
CONTINUOUS_FEATURE_COLUMNS = TARGET_COLUMNS.copy()
BINARY_FEATURE_COLUMNS = ACTUATOR_COLUMNS.copy()

assert len(FEATURE_COLUMNS) == 8
assert len(TARGET_COLUMNS) == 5
assert set(FEATURE_COLUMNS) == set(CONTINUOUS_FEATURE_COLUMNS + BINARY_FEATURE_COLUMNS)
assert "timestamp" not in FEATURE_COLUMNS

## 11 - Scaling Policy

Continuous sensors dung `StandardScaler`; actuator 0/1 duoc passthrough. Feature va target co scaler
rieng de prediction scaled co the inverse-transform ve don vi vat ly ma khong lam bien dang control state.

In [ ]:
@dataclass
class FeatureTransformer:
    """Scale continuous sensor columns and preserve binary actuator columns."""

    continuous_scaler: StandardScaler
    feature_columns: list[str]
    continuous_columns: list[str]
    binary_columns: list[str]

    def transform_window(self, values: np.ndarray) -> np.ndarray:
        transformed = np.asarray(values, dtype=np.float32).copy()
        continuous_positions = [self.feature_columns.index(name) for name in self.continuous_columns]
        binary_positions = [self.feature_columns.index(name) for name in self.binary_columns]
        transformed[:, continuous_positions] = self.continuous_scaler.transform(
            transformed[:, continuous_positions]
        ).astype(np.float32)
        binary_values = transformed[:, binary_positions]
        if not np.isin(binary_values, [0.0, 1.0]).all():
            raise ValueError("Binary actuator passthrough received a non-binary value")
        return transformed


def valid_target_positions(
    frame: pd.DataFrame,
    target_start: pd.Timestamp,
    target_end: pd.Timestamp,
    lookback_steps: int,
    forecast_horizon: int,
) -> np.ndarray:
    """Return target positions with continuous, past-only lookback context."""
    timestamps = frame["timestamp"].to_numpy(dtype="datetime64[ns]")
    in_split = frame["timestamp"].between(target_start, target_end).to_numpy()
    positions = np.flatnonzero(in_split).astype(np.int64)
    input_end = positions - forecast_horizon
    input_start = input_end - lookback_steps + 1
    feasible = input_start >= 0
    positions, input_end, input_start = positions[feasible], input_end[feasible], input_start[feasible]
    if not len(positions):
        return positions.astype(np.int32)
    expected_input_span = np.timedelta64(lookback_steps - 1, "h")
    expected_horizon = np.timedelta64(forecast_horizon, "h")
    continuous = (
        (timestamps[input_end] - timestamps[input_start] == expected_input_span)
        & (timestamps[positions] - timestamps[input_end] == expected_horizon)
    )
    return positions[continuous].astype(np.int32)

## 12 - Fit Train-only Scalers

Scaler chi quan sat development scenarios trong khoang TRAIN. Validation, 2025 va held-out scenarios
khong tham gia `fit`; tat ca cac split sau chi goi `transform` bang cung train-fitted statistics.

In [ ]:
def fit_train_only_scalers(
    frames: dict[str, pd.DataFrame],
    scenario_ids: list[str],
    target_start: pd.Timestamp,
    target_end: pd.Timestamp,
) -> tuple[FeatureTransformer, StandardScaler, dict[str, int]]:
    """Fit feature and target scalers exclusively from development TRAIN observations."""
    feature_scaler = StandardScaler()
    target_scaler = StandardScaler()
    feature_rows = 0
    target_rows = 0
    for scenario_id in scenario_ids:
        frame = frames[scenario_id]
        train_mask = frame["timestamp"].between(target_start, target_end)
        train_features = frame.loc[train_mask, CONTINUOUS_FEATURE_COLUMNS].to_numpy(np.float64)
        if not len(train_features):
            raise ValueError(f"{scenario_id}: no rows available for scaler fitting")
        feature_scaler.partial_fit(train_features)
        feature_rows += len(train_features)

        target_positions = valid_target_positions(
            frame, target_start, target_end, LOOKBACK_STEPS, FORECAST_HORIZON
        )
        train_targets = frame.iloc[target_positions][TARGET_COLUMNS].to_numpy(np.float64)
        target_scaler.partial_fit(train_targets)
        target_rows += len(train_targets)

    transformer = FeatureTransformer(
        continuous_scaler=feature_scaler,
        feature_columns=FEATURE_COLUMNS,
        continuous_columns=CONTINUOUS_FEATURE_COLUMNS,
        binary_columns=BINARY_FEATURE_COLUMNS,
    )
    return transformer, target_scaler, {
        "feature_fit_rows": feature_rows,
        "target_fit_rows": target_rows,
    }


feature_transformer, target_scaler, scaler_fit_audit = fit_train_only_scalers(
    scenario_frames, active_development_ids, train_start, train_end
)
assert int(feature_transformer.continuous_scaler.n_samples_seen_) == scaler_fit_audit["feature_fit_rows"]
assert int(target_scaler.n_samples_seen_) == scaler_fit_audit["target_fit_rows"]
print(f"Train-only scaler audit: {scaler_fit_audit}")

## 13 - Sliding Window Definition

Voi `lookback=24`, input gom 24 gio ket thuc tai `t`; target o `t+1`. Compact index chi luu scenario
code va target position, khong materialize mang khong lo `N x 24 x 8`.

In [ ]:
@dataclass(frozen=True)
class SequenceIndex:
    """Compact references to valid targets inside independent scenario trajectories."""

    split_name: str
    scenario_ids: tuple[str, ...]
    scenario_codes: np.ndarray
    target_positions: np.ndarray
    target_start: pd.Timestamp
    target_end: pd.Timestamp

    def __len__(self) -> int:
        return int(len(self.target_positions))

    def resolve(self, item: int) -> tuple[str, int]:
        return self.scenario_ids[int(self.scenario_codes[item])], int(self.target_positions[item])


def build_sequence_index(
    frames: dict[str, pd.DataFrame],
    scenario_ids: list[str],
    split_name: str,
    target_start: pd.Timestamp,
    target_end: pd.Timestamp,
) -> SequenceIndex:
    """Build a compact target index while preserving each scenario boundary."""
    ordered_ids = tuple(sorted(scenario_ids))
    code_chunks: list[np.ndarray] = []
    position_chunks: list[np.ndarray] = []
    for scenario_code, scenario_id in enumerate(ordered_ids):
        positions = valid_target_positions(
            frames[scenario_id], target_start, target_end, LOOKBACK_STEPS, FORECAST_HORIZON
        )
        code_chunks.append(np.full(len(positions), scenario_code, dtype=np.int16))
        position_chunks.append(positions)
    return SequenceIndex(
        split_name=split_name,
        scenario_ids=ordered_ids,
        scenario_codes=np.concatenate(code_chunks) if code_chunks else np.empty(0, dtype=np.int16),
        target_positions=(
            np.concatenate(position_chunks) if position_chunks else np.empty(0, dtype=np.int32)
        ),
        target_start=target_start,
        target_end=target_end,
    )

## 14 - Sequence Index Generation

Tao rieng index cho TRAIN, validation, temporal test, scenario test va combined test. Target quyet dinh
split; historical context ngay truoc boundary duoc phep vi do la thong tin da co tai inference time.

In [ ]:
train_sequence_index = build_sequence_index(
    scenario_frames, active_development_ids, "train", train_start, train_end
)
validation_sequence_index = build_sequence_index(
    scenario_frames, active_development_ids, "validation", validation_start, validation_end
)
temporal_test_sequence_index = build_sequence_index(
    scenario_frames, active_development_ids, "temporal_test", temporal_test_start, temporal_test_end
)
scenario_test_sequence_index = build_sequence_index(
    scenario_frames,
    active_held_out_ids,
    "scenario_test_seen_time",
    train_start,
    validation_end,
)
combined_test_sequence_index = build_sequence_index(
    scenario_frames,
    active_held_out_ids,
    "combined_scenario_temporal_test",
    temporal_test_start,
    temporal_test_end,
)

sequence_indices = {
    "train": train_sequence_index,
    "validation": validation_sequence_index,
    "temporal_test": temporal_test_sequence_index,
    "scenario_test": scenario_test_sequence_index,
    "combined_test": combined_test_sequence_index,
}
if any(len(index) == 0 for index in sequence_indices.values()):
    raise ValueError("At least one required split has no valid sequence windows")
print({name: len(index) for name, index in sequence_indices.items()})

## 15 - Lazy GreenhouseSequenceDataset

Dataset dinh nghia mot sample: no resolve scenario, cat lookback, transform feature, lay future target va
tra `float32` tensor. Tensor khong duoc dua len CUDA trong Dataset.

In [ ]:
class GreenhouseSequenceDataset(Dataset):
    """Lazy PyTorch dataset over compact sequence references."""

    def __init__(
        self,
        frames: dict[str, pd.DataFrame],
        sequence_index: SequenceIndex,
        feature_transformer: FeatureTransformer,
        target_scaler: StandardScaler,
    ) -> None:
        self.frames = frames
        self.sequence_index = sequence_index
        self.feature_transformer = feature_transformer
        self.target_scaler = target_scaler

    def __len__(self) -> int:
        return len(self.sequence_index)

    def __getitem__(self, item: int) -> tuple[torch.Tensor, torch.Tensor]:
        scenario_id, target_position = self.sequence_index.resolve(item)
        frame = self.frames[scenario_id]
        input_end = target_position - FORECAST_HORIZON
        input_start = input_end - LOOKBACK_STEPS + 1
        raw_window = frame.iloc[input_start : input_end + 1][FEATURE_COLUMNS].to_numpy(np.float32)
        if raw_window.shape != (LOOKBACK_STEPS, len(FEATURE_COLUMNS)):
            raise RuntimeError(f"Invalid window shape for {scenario_id}: {raw_window.shape}")
        scaled_window = self.feature_transformer.transform_window(raw_window)
        raw_target = frame.iloc[[target_position]][TARGET_COLUMNS].to_numpy(np.float32)
        scaled_target = self.target_scaler.transform(raw_target).astype(np.float32)[0]
        return torch.from_numpy(scaled_window), torch.from_numpy(scaled_target)


datasets = {
    name: GreenhouseSequenceDataset(
        scenario_frames, index, feature_transformer, target_scaler
    )
    for name, index in sequence_indices.items()
}

## 16 - DataLoader Construction

DataLoader gom sample thanh batch. Chi training windows duoc shuffle sau khi split da dung; validation
va test giu thu tu. Dieu nay khac hoan toan voi random split raw rows, von se gay leakage.

In [ ]:
def seed_worker(worker_id: int) -> None:
    """Derive deterministic NumPy/Python seeds for each optional DataLoader worker."""
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_loader(dataset: Dataset, shuffle: bool) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        worker_init_fn=seed_worker if NUM_WORKERS else None,
        generator=data_loader_generator if shuffle else None,
        persistent_workers=NUM_WORKERS > 0,
        drop_last=False,
    )


train_loader = make_loader(datasets["train"], shuffle=True)
val_loader = make_loader(datasets["validation"], shuffle=False)
test_temporal_loader = make_loader(datasets["temporal_test"], shuffle=False)
test_scenario_loader = make_loader(datasets["scenario_test"], shuffle=False)
test_combined_loader = make_loader(datasets["combined_test"], shuffle=False)

## 17 - Boundary & Leakage Checks

Kiem tra vectorized rang moi target nam trong dung split, input lien tuc, hoan toan o qua khu va cung
scenario. Target dau 2024/2025 phai dung duoc context truoc boundary ma khong dung future information.

In [ ]:
def assert_sequence_integrity(
    index: SequenceIndex,
    frames: dict[str, pd.DataFrame],
) -> dict[str, int]:
    """Assert split membership, continuity and past-only context for every indexed sequence."""
    checked = 0
    for scenario_code, scenario_id in enumerate(index.scenario_ids):
        positions = index.target_positions[index.scenario_codes == scenario_code].astype(np.int64)
        if not len(positions):
            continue
        frame = frames[scenario_id]
        timestamps = frame["timestamp"].to_numpy(dtype="datetime64[ns]")
        input_end = positions - FORECAST_HORIZON
        input_start = input_end - LOOKBACK_STEPS + 1
        targets = timestamps[positions]
        if not ((targets >= np.datetime64(index.target_start)) & (targets <= np.datetime64(index.target_end))).all():
            raise AssertionError(f"{index.split_name}: target outside split")
        if not (timestamps[input_end] < targets).all():
            raise AssertionError(f"{index.split_name}: future leakage")
        if not (
            timestamps[input_end] - timestamps[input_start]
            == np.timedelta64(LOOKBACK_STEPS - 1, "h")
        ).all():
            raise AssertionError(f"{index.split_name}: discontinuous/cross-boundary lookback")
        if not (
            targets - timestamps[input_end] == np.timedelta64(FORECAST_HORIZON, "h")
        ).all():
            raise AssertionError(f"{index.split_name}: incorrect target alignment")
        checked += len(positions)
    if checked != len(index):
        raise AssertionError(f"{index.split_name}: sequence audit count mismatch")
    return {"checked_windows": checked, "scenario_count": len(index.scenario_ids)}


leakage_audit = {
    name: assert_sequence_integrity(index, scenario_frames)
    for name, index in sequence_indices.items()
}


def assert_boundary_context(index: SequenceIndex, boundary: pd.Timestamp) -> None:
    for scenario_code, scenario_id in enumerate(index.scenario_ids):
        positions = index.target_positions[index.scenario_codes == scenario_code]
        frame = scenario_frames[scenario_id]
        boundary_positions = positions[
            frame.iloc[positions]["timestamp"].to_numpy() == np.datetime64(boundary)
        ]
        if len(boundary_positions) != 1:
            raise AssertionError(f"{scenario_id}: boundary target {boundary} missing")
        input_end = int(boundary_positions[0]) - FORECAST_HORIZON
        input_start = input_end - LOOKBACK_STEPS + 1
        if not frame.iloc[input_start]["timestamp"] < boundary:
            raise AssertionError(f"{scenario_id}: boundary context was not carried from history")


assert_boundary_context(validation_sequence_index, validation_start)
assert_boundary_context(temporal_test_sequence_index, temporal_test_start)
print(f"Leakage and boundary checks passed: {leakage_audit}")

## 18 - Smoke Test

Smoke test chi xac nhan pipeline end-to-end: dtype, finite values, shapes, binary passthrough, alignment,
inverse transform va nhieu batch. No khong do accuracy va khong train model.

In [ ]:
def run_pipeline_smoke_test() -> dict[str, object]:
    batches = []
    train_iterator = iter(train_loader)
    for _ in range(2):
        batches.append(next(train_iterator))
    for features, targets in batches:
        assert features.dtype == torch.float32
        assert targets.dtype == torch.float32
        assert features.shape[1:] == (LOOKBACK_STEPS, len(FEATURE_COLUMNS))
        assert targets.shape[1:] == (len(TARGET_COLUMNS),)
        assert torch.isfinite(features).all()
        assert torch.isfinite(targets).all()
        actuator_values = features[:, :, -len(BINARY_FEATURE_COLUMNS) :].numpy()
        assert np.isin(actuator_values, [0.0, 1.0]).all()

    sample_features, sample_target = datasets["train"][0]
    scenario_id, target_position = train_sequence_index.resolve(0)
    restored_target = target_scaler.inverse_transform(sample_target.numpy()[None, :])[0]
    raw_target = scenario_frames[scenario_id].iloc[target_position][TARGET_COLUMNS].to_numpy(np.float32)
    np.testing.assert_allclose(restored_target, raw_target, rtol=1e-5, atol=1e-5)

    input_end = target_position - FORECAST_HORIZON
    input_start = input_end - LOOKBACK_STEPS + 1
    raw_binary = scenario_frames[scenario_id].iloc[input_start : input_end + 1][BINARY_FEATURE_COLUMNS].to_numpy(np.float32)
    np.testing.assert_array_equal(sample_features.numpy()[:, -3:], raw_binary)

    validation_batch = next(iter(val_loader))
    temporal_batch = next(iter(test_temporal_loader))
    scenario_batch = next(iter(test_scenario_loader))
    combined_batch = next(iter(test_combined_loader))
    return {
        "scenarios_used": active_scenario_ids,
        "train_batch_shape": list(batches[0][0].shape),
        "target_batch_shape": list(batches[0][1].shape),
        "validation_batch_shape": list(validation_batch[0].shape),
        "temporal_batch_shape": list(temporal_batch[0].shape),
        "scenario_batch_shape": list(scenario_batch[0].shape),
        "combined_batch_shape": list(combined_batch[0].shape),
        "finite": True,
        "inverse_transform": "PASS",
        "binary_passthrough": "PASS",
    }


smoke_test_result = run_pipeline_smoke_test()
print(f"Pipeline smoke validation PASS: {smoke_test_result}")

## 19 - Dataset / Window Statistics

Bang nhe nay cho biet raw rows dang active, so window moi split va train statistics. No ho tro audit ma
khong tao visualization hoac ban sao window lon.

In [ ]:
scenario_split_summary = pd.DataFrame(
    {
        "split": ["development", "held_out"],
        "scenario_count": [len(development_scenario_ids), len(held_out_scenario_ids)],
        "scenario_ids": [development_scenario_ids, held_out_scenario_ids],
    }
)
window_statistics = pd.DataFrame(
    [{"split": name, "windows": len(index)} for name, index in sequence_indices.items()]
)
train_feature_statistics = pd.DataFrame(
    {
        "feature": CONTINUOUS_FEATURE_COLUMNS,
        "mean": feature_transformer.continuous_scaler.mean_,
        "scale": feature_transformer.continuous_scaler.scale_,
    }
)
print(scenario_split_summary.to_string(index=False))
print(window_statistics.to_string(index=False))
print(train_feature_statistics.to_string(index=False))

## 20 - Export Preprocessing Artifacts

Export scaler va manifest/config de notebook training sau dung dung split va transformation. Artifact
smoke duoc tach directory; khong co model checkpoint trong milestone nay.

In [ ]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(feature_transformer.continuous_scaler, ARTIFACT_DIR / "feature_scaler.pkl")
joblib.dump(target_scaler, ARTIFACT_DIR / "target_scaler.pkl")

split_manifest = {
    "seed": SEED,
    "development_scenario_ids": development_scenario_ids,
    "held_out_scenario_ids": held_out_scenario_ids,
    "train_date_range": [str(FULL_SPLIT_RANGES["train"][0]), str(FULL_SPLIT_RANGES["train"][1])],
    "validation_date_range": [str(FULL_SPLIT_RANGES["validation"][0]), str(FULL_SPLIT_RANGES["validation"][1])],
    "temporal_test_date_range": [str(FULL_SPLIT_RANGES["temporal_test"][0]), str(FULL_SPLIT_RANGES["temporal_test"][1])],
    "lookback_steps": LOOKBACK_STEPS,
    "forecast_horizon": FORECAST_HORIZON,
    "scenario_selection_strategy": "deterministic normalized-parameter farthest-point selection",
    "canonical_index": str(INDEX_FILE),
}
preprocessing_config = {
    "feature_columns": FEATURE_COLUMNS,
    "target_columns": TARGET_COLUMNS,
    "continuous_feature_columns": CONTINUOUS_FEATURE_COLUMNS,
    "binary_feature_columns": BINARY_FEATURE_COLUMNS,
    "feature_scaling_strategy": "StandardScaler on continuous sensors; binary passthrough",
    "target_scaling_strategy": "StandardScaler",
    "binary_actuator_policy": "passthrough_0_1",
    "scaler_fit_source": "development scenarios, TRAIN period only",
    "lookback_steps": LOOKBACK_STEPS,
    "forecast_horizon": FORECAST_HORIZON,
    "smoke_test_execution": SMOKE_TEST,
}
(ARTIFACT_DIR / "split_manifest.json").write_text(
    json.dumps(split_manifest, indent=2), encoding="utf-8"
)
(ARTIFACT_DIR / "preprocessing_config.json").write_text(
    json.dumps(preprocessing_config, indent=2), encoding="utf-8"
)
print(f"Preprocessing artifacts exported to {ARTIFACT_DIR.resolve()}")

## 21 - Final Pipeline Summary

Pipeline da san sang o muc DataLoader: canonical membership, split hai chieu, train-only scaling, lazy
windows va leakage checks deu co assertion. Milestone ke tiep moi duoc dinh nghia model va training loop.

In [ ]:
pipeline_summary = {
    "status": "PASS",
    "canonical_scenarios": len(canonical_index),
    "canonical_rows": int(canonical_index["ml_rows"].astype(int).sum()),
    "development_scenarios": len(development_scenario_ids),
    "held_out_scenarios": len(held_out_scenario_ids),
    "feature_count": len(FEATURE_COLUMNS),
    "target_count": len(TARGET_COLUMNS),
    "lookback_steps": LOOKBACK_STEPS,
    "forecast_horizon": FORECAST_HORIZON,
    "window_counts": {name: len(index) for name, index in sequence_indices.items()},
    "scaler_policy": "TRAIN only",
    "actuator_policy": "binary passthrough",
    "scenario_boundary_check": "PASS",
    "temporal_leakage_check": "PASS",
    "smoke_validation": smoke_test_result,
    "full_training_executed": False,
}
print(json.dumps(pipeline_summary, indent=2))